# Fase 2: política y métricas reproducibles
Ejecutar primero `python -m data_pipeline.run_pipeline --full`. Este notebook recalcula desde silver sin conectarse a AWS ni abrir `.env`. La política se toma de la corrida exitosa.

Calibración: julio 2025–marzo 2026. Prueba: abril–17 junio 2026. El test no selecciona umbrales. Los datos son sintéticos; resultados offline, sin garantías de producción.

In [ ]:
import json, sys
from pathlib import Path
import duckdb, pandas as pd, yaml
ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(ROOT))
from data_pipeline.analytics import analyze
from data_pipeline.lineage import write_json
manifest = json.loads((ROOT / 'data/manifest.json').read_text(encoding='utf-8'))
run_dir = ROOT / 'data/runs' / manifest['run_id']
policy = yaml.safe_load((run_dir / 'policy.yaml').read_text(encoding='utf-8'))
con = duckdb.connect()
for path in sorted((run_dir / 'silver').glob('*.parquet')):
    con.read_parquet(str(path)).create_view(path.stem)
output = ROOT / 'reports/notebook_policy'
calibration, metrics = analyze(con, output, policy)
write_json(ROOT / 'analysis/metricas_problema.json', metrics)
print('Corrida:', manifest['run_id'], 'Política:', policy['policy_version'])


## Fraude
Candidato empírico: menor corte entero con precisión ≥30 % (bajo) o ≥99 % (alto), y ≥20 predicciones en calibración. Reportamos también los cortes acordados 30/40. El recall incluye TODOS los fraudes, incluso los de score nulo. `predicted` es el denominador de precisión.

In [ ]:
comparison = []
for split, data in calibration['fraud'].items():
    for row in data['thresholds']:
        if row['threshold'] in (30, 31, 40):
            comparison.append(dict(split=split, n=data['n'], fraud_n=data['fraud_n'], null_score_n=data['null_score_n'], **row))
display(pd.DataFrame(comparison))
display(pd.DataFrame(calibration['amount_calibration']))
display(pd.DataFrame(calibration['amount_test']))


## Decisión adoptada
Se conservan las bandas acordadas 30/40 y se actualizan los límites de monto al p95 de calibración (centavos). La alternativa ≥31 tiene mejor recall en esta muestra, pero cambiaría el contrato de comportamiento documentado y el test de backend que exige R9 con score=35; se coordinará antes de adoptarla. Se registra la alternativa, sin ocultarla ni usar el test para escogerla.
La ventana de reclamo de 60 días y los SLA son política sintética: estos datos no justifican una ventana legal. El motor completo tiene precedencia de reglas; estas métricas del score no son métricas end-to-end.

In [ ]:
display(pd.DataFrame(metrics['call_center_by_category']))
display(pd.DataFrame([metrics['complaints']]))
display(pd.DataFrame([metrics['disputes']]))
display(pd.DataFrame(metrics['csat_by_contact_reason']))
display(pd.DataFrame(metrics['transactions_by_currency']))
con.close()


## Denominadores y limitaciones
Disputas / todas las quejas y disputas / quejas con subcategoría son porcentajes diferentes. Días de resolución usan únicamente los valores observados; no asumir que son casos cerrados. CSAT se filtra por survey_type y excluye NPS/CES. El histórico de contactos no es la misma carga de evaluación conversacional de la fase 6. Timestamps sin offset se interpretan según el supuesto explícito del manifiesto; business_date siempre conserva la partición.